In [28]:
import pandas as pd 
import numpy as np
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

In [29]:
knowledge_base = [
    {
        "category": "Service Policy",
        "title": "Support Channels",
        "text": "Customers can contact customer support through approved channels such as email, phone, and the online support portal."
    },
    {
        "category": "Service Policy",
        "title": "Business Hours",
        "text": "Standard customer support requests are normally handled during business hours."
    },
    {
        "category": "Service Policy",
        "title": "Urgent Issues",
        "text": "Urgent service interruptions may receive higher priority when they have a significant impact on the customer."
    },

    {
        "category": "Customer Support FAQ",
        "title": "Required Information",
        "text": "Customers should provide their customer ID, order number, and a clear description of the issue when contacting support."
    },
    {
        "category": "Customer Support FAQ",
        "title": "Product Not Working",
        "text": "If a product is not working, customers should first restart the product and check that required connections are secure."
    },
    {
        "category": "Customer Support FAQ",
        "title": "Software Problems",
        "text": "When reporting a software problem, customers should provide the error message and explain when the problem started."
    },

    {
        "category": "Contract Information",
        "title": "Contract Review",
        "text": "Customers should review their applicable contract terms before changing or cancelling a service."
    },
    {
        "category": "Contract Information",
        "title": "Month-to-Month Contract",
        "text": "Month-to-month contracts generally provide greater flexibility because the customer has a shorter commitment period."
    },
    {
        "category": "Contract Information",
        "title": "Contract Changes",
        "text": "Contract changes must be processed according to approved business procedures and applicable agreement terms."
    },

    {
        "category": "Cancellation Policy",
        "title": "Cancellation Request",
        "text": "Customers who want to cancel a service should contact customer support through an approved support channel."
    },
    {
        "category": "Cancellation Policy",
        "title": "Account Verification",
        "text": "The customer account should be verified before a cancellation request is processed."
    },
    {
        "category": "Cancellation Policy",
        "title": "Cancellation Conditions",
        "text": "Applicable contract terms should be reviewed to determine whether cancellation conditions or charges apply."
    },

    {
        "category": "Billing Policy",
        "title": "Billing Questions",
        "text": "Customers can contact support when they have questions about their invoice, payment, billing amount, or unexpected charge."
    },
    {
        "category": "Billing Policy",
        "title": "Billing Dispute",
        "text": "Customers submitting a billing dispute should identify the specific charge they believe is incorrect."
    },
    {
        "category": "Billing Policy",
        "title": "Billing Adjustment",
        "text": "If an incorrect charge is identified, the appropriate billing adjustment should follow applicable approval rules."
    },

    {
        "category": "Retention Guidelines",
        "title": "Cancellation Reason",
        "text": "When a customer considers cancellation, support staff should first understand the customer's reason for considering cancellation."
    },
    {
        "category": "Retention Guidelines",
        "title": "Technical Retention",
        "text": "For technical problems, support staff should attempt appropriate troubleshooting or escalation before discussing cancellation."
    },
    {
        "category": "Retention Guidelines",
        "title": "Customer Choice",
        "text": "Customers should receive clear information about available options and should not be pressured into continuing a service."
    },

    {
        "category": "Support Procedures",
        "title": "Record Support Requests",
        "text": "All customer support requests should be recorded in the support system so that the issue and its progress can be tracked."
    },
    {
        "category": "Support Procedures",
        "title": "Technical Troubleshooting",
        "text": "For technical issues, support staff should collect relevant details and record troubleshooting steps already completed."
    },
    {
        "category": "Support Procedures",
        "title": "Issue Escalation",
        "text": "Issues that cannot be resolved using standard procedures should be escalated to the appropriate support team."
    },

    {
        "category": "Business Rules",
        "title": "Account Verification",
        "text": "Support staff should verify the customer account before processing requests involving billing, contracts, or cancellations."
    },
    {
        "category": "Business Rules",
        "title": "Refund Approval",
        "text": "Refunds and billing adjustments should only be provided when the applicable eligibility requirements are satisfied."
    },
    {
        "category": "Business Rules",
        "title": "Unauthorized Exceptions",
        "text": "Support staff should not create unauthorized exceptions to approved business rules or service policies."
    }
]

kb_df = pd.DataFrame(knowledge_base)

print("Knowledge Base shape:", kb_df.shape)
print(kb_df.head())

Knowledge Base shape: (24, 3)
               category                 title  \
0        Service Policy      Support Channels   
1        Service Policy        Business Hours   
2        Service Policy         Urgent Issues   
3  Customer Support FAQ  Required Information   
4  Customer Support FAQ   Product Not Working   

                                                text  
0  Customers can contact customer support through...  
1  Standard customer support requests are normall...  
2  Urgent service interruptions may receive highe...  
3  Customers should provide their customer ID, or...  
4  If a product is not working, customers should ...  


In [30]:
service_policy_records = [
    {
        "category": "Service Policy",
        "title": "Support Request Priority",
        "text": "Support requests should normally be handled in the order they are received unless an issue requires higher priority."
    },
    {
        "category": "Service Policy",
        "title": "Customer Information",
        "text": "Customers may be asked to provide relevant account or order information so that support staff can investigate their request."
    },
    {
        "category": "Service Policy",
        "title": "Account Verification",
        "text": "Account information should be verified before support staff discuss sensitive billing or account details."
    },
    {
        "category": "Service Policy",
        "title": "Technical Support",
        "text": "Customers reporting technical problems may be asked to complete basic troubleshooting steps before escalation."
    },
    {
        "category": "Service Policy",
        "title": "Support Case Updates",
        "text": "Customers can request an update about an open support case through an approved support channel."
    },
    {
        "category": "Service Policy",
        "title": "Existing Support Case",
        "text": "Customers reporting an existing issue should provide the previous support reference number when available."
    },
    {
        "category": "Service Policy",
        "title": "Support Documentation",
        "text": "Support staff should record the reported problem, troubleshooting actions, customer communications, and final resolution."
    },
    {
        "category": "Service Policy",
        "title": "Escalation Process",
        "text": "Issues that cannot be resolved through standard support procedures should be escalated to the appropriate team."
    },
    {
        "category": "Service Policy",
        "title": "Clear Communication",
        "text": "Support staff should communicate instructions and service decisions clearly so that customers understand the available options."
    },
    {
        "category": "Service Policy",
        "title": "Sensitive Information",
        "text": "Customers should avoid sharing unnecessary sensitive information when submitting a support request."
    }
]

knowledge_base.extend(service_policy_records)

print("Total Knowledge Base documents:", len(knowledge_base))

Total Knowledge Base documents: 34


In [31]:
support_faq_records = [
    {
        "category": "Customer Support FAQ",
        "title": "How to Contact Support",
        "text": "Customers can contact support through approved email, phone, or online support channels."
    },
    {
        "category": "Customer Support FAQ",
        "title": "What Information to Provide",
        "text": "Customers should provide their customer ID, order number, and a clear description of the issue."
    },
    {
        "category": "Customer Support FAQ",
        "title": "Product Restart",
        "text": "If a product is not working, customers should restart the product before trying additional troubleshooting steps."
    },
    {
        "category": "Customer Support FAQ",
        "title": "Connection Check",
        "text": "Customers experiencing product problems should check that all required connections are secure."
    },
    {
        "category": "Customer Support FAQ",
        "title": "Error Messages",
        "text": "Customers reporting software problems should provide any error message displayed by the product."
    },
    {
        "category": "Customer Support FAQ",
        "title": "When the Problem Started",
        "text": "Customers should explain when a technical problem started because this information can help support investigate the issue."
    },
    {
        "category": "Customer Support FAQ",
        "title": "Previous Troubleshooting",
        "text": "Customers should tell support staff which troubleshooting steps they have already tried."
    },
    {
        "category": "Customer Support FAQ",
        "title": "Existing Support Reference",
        "text": "Customers contacting support about an existing issue should provide the previous support reference number when available."
    },
    {
        "category": "Customer Support FAQ",
        "title": "Account Verification",
        "text": "Support staff may need to verify the customer account before discussing account-specific information."
    },
    {
        "category": "Customer Support FAQ",
        "title": "Sensitive Information",
        "text": "Customers should avoid including unnecessary sensitive information in support messages."
    }
]

knowledge_base.extend(support_faq_records)

print("Total Knowledge Base documents:", len(knowledge_base))

Total Knowledge Base documents: 44


In [32]:
contract_records = [
    {
        "category": "Contract Information",
        "title": "Contract Type",
        "text": "Customers should review their current contract type before making decisions about changing or cancelling a service."
    },
    {
        "category": "Contract Information",
        "title": "Month-to-Month Contract",
        "text": "Month-to-month contracts generally provide greater flexibility because they involve a shorter commitment period."
    },
    {
        "category": "Contract Information",
        "title": "One-Year Contract",
        "text": "One-year contracts provide a longer service commitment and may have conditions that differ from month-to-month agreements."
    },
    {
        "category": "Contract Information",
        "title": "Two-Year Contract",
        "text": "Two-year contracts involve a longer service commitment and customers should review the applicable contract conditions."
    },
    {
        "category": "Contract Information",
        "title": "Contract Clarification",
        "text": "Customers can contact support if they need clarification about their contract type or service commitment."
    },
    {
        "category": "Contract Information",
        "title": "Contract Cancellation",
        "text": "Customers should review their contract terms before requesting cancellation because applicable conditions may affect the request."
    },
    {
        "category": "Contract Information",
        "title": "Early Termination",
        "text": "Early termination conditions may depend on the customer's applicable agreement and should be checked before processing termination."
    },
    {
        "category": "Contract Information",
        "title": "Contract Changes",
        "text": "Contract changes should be processed according to the applicable agreement and approved business procedures."
    },
    {
        "category": "Contract Information",
        "title": "Contract Authorization",
        "text": "Support staff should not promise contract changes, discounts, or exceptions without the required authorization."
    },
    {
        "category": "Contract Information",
        "title": "Account-Specific Contract Details",
        "text": "Support staff should verify the customer account before discussing account-specific contract information."
    }
]

knowledge_base.extend(contract_records)

print("Total Knowledge Base documents:", len(knowledge_base))

Total Knowledge Base documents: 54


In [33]:
cancellation_records = [
    {
        "category": "Cancellation Policy",
        "title": "Cancellation Request",
        "text": "Customers who want to cancel a service should contact customer support through an approved support channel."
    },
    {
        "category": "Cancellation Policy",
        "title": "Verify Account",
        "text": "The customer account should be verified before a cancellation request is processed."
    },
    {
        "category": "Cancellation Policy",
        "title": "Review Contract",
        "text": "Support staff should review the customer's applicable contract before processing a cancellation request."
    },
    {
        "category": "Cancellation Policy",
        "title": "Cancellation Conditions",
        "text": "Applicable contract terms should be checked to determine whether cancellation conditions or charges apply."
    },
    {
        "category": "Cancellation Policy",
        "title": "Explain Charges",
        "text": "Customers should be informed about applicable cancellation charges or remaining commitments before cancellation is completed."
    },
    {
        "category": "Cancellation Policy",
        "title": "Cancellation Reason",
        "text": "Support staff should record the customer's reason for cancellation in the support system."
    },
    {
        "category": "Cancellation Policy",
        "title": "Cancellation Date",
        "text": "The cancellation date should be recorded accurately in the customer account after the cancellation is processed."
    },
    {
        "category": "Cancellation Policy",
        "title": "Withdraw Cancellation",
        "text": "If a customer changes their mind before cancellation is completed, support should check whether the request can still be withdrawn."
    },
    {
        "category": "Cancellation Policy",
        "title": "Troubleshooting Before Cancellation",
        "text": "When applicable, customers experiencing service problems may be offered appropriate troubleshooting information before cancellation."
    },
    {
        "category": "Cancellation Policy",
        "title": "Final Resolution",
        "text": "The final outcome of a cancellation request should be documented in the support system."
    }
]

knowledge_base.extend(cancellation_records)

print("Total Knowledge Base documents:", len(knowledge_base))

Total Knowledge Base documents: 64


In [34]:
billing_records = [
    {
        "category": "Billing Policy",
        "title": "Invoice Questions",
        "text": "Customers can contact support when they have questions about their monthly invoice or billing amount."
    },
    {
        "category": "Billing Policy",
        "title": "Unexpected Charge",
        "text": "Customers who notice an unexpected charge should identify the specific charge when contacting support."
    },
    {
        "category": "Billing Policy",
        "title": "Billing Period",
        "text": "Customers submitting a billing dispute should provide the relevant billing period when possible."
    },
    {
        "category": "Billing Policy",
        "title": "Review Invoice",
        "text": "Customers should review their invoice before submitting a billing dispute so that the issue can be clearly understood."
    },
    {
        "category": "Billing Policy",
        "title": "Billing Dispute Review",
        "text": "Support staff should record a billing dispute and review the available billing information."
    },
    {
        "category": "Billing Policy",
        "title": "Incorrect Charge",
        "text": "If an incorrect charge is identified, the appropriate billing adjustment should follow the applicable approval rules."
    },
    {
        "category": "Billing Policy",
        "title": "Refund Eligibility",
        "text": "A disputed charge does not automatically qualify for a refund and must be reviewed according to eligibility requirements."
    },
    {
        "category": "Billing Policy",
        "title": "Approved Refund",
        "text": "Approved refunds and billing adjustments should be recorded in the customer's account."
    },
    {
        "category": "Billing Policy",
        "title": "Billing Decision",
        "text": "Support staff should communicate billing decisions clearly to customers after reviewing the request."
    },
    {
        "category": "Billing Policy",
        "title": "Account Verification",
        "text": "Customer account information should be verified before discussing account-specific billing information."
    }
]

knowledge_base.extend(billing_records)

print("Total Knowledge Base documents:", len(knowledge_base))

Total Knowledge Base documents: 74


In [35]:
retention_records = [
    {
        "category": "Retention Guidelines",
        "title": "Identify At-Risk Customers",
        "text": "Customers showing signs of dissatisfaction, repeated support issues, or cancellation intent should be considered for retention actions."
    },
    {
        "category": "Retention Guidelines",
        "title": "Understand Customer Concerns",
        "text": "Support staff should understand the customer's main concern before recommending a retention action."
    },
    {
        "category": "Retention Guidelines",
        "title": "Review Customer History",
        "text": "Customer history and previous support interactions can be reviewed to understand recurring problems."
    },
    {
        "category": "Retention Guidelines",
        "title": "Prioritize High-Risk Customers",
        "text": "Customers identified as high churn risk should receive appropriate retention attention."
    },
    {
        "category": "Retention Guidelines",
        "title": "Address Service Issues",
        "text": "When service problems contribute to dissatisfaction, resolving the underlying issue should be prioritized."
    },
    {
        "category": "Retention Guidelines",
        "title": "Explain Available Options",
        "text": "Customers should be informed about relevant service or account options that may address their concerns."
    },
    {
        "category": "Retention Guidelines",
        "title": "Avoid Unnecessary Offers",
        "text": "Retention offers should be relevant to the customer's situation and should not be provided without understanding the problem."
    },
    {
        "category": "Retention Guidelines",
        "title": "Escalate Complex Cases",
        "text": "Complex retention cases should be escalated to the appropriate support or retention team."
    },
    {
        "category": "Retention Guidelines",
        "title": "Record Retention Actions",
        "text": "Retention actions and customer responses should be recorded for future reference."
    },
    {
        "category": "Retention Guidelines",
        "title": "Respect Cancellation Decisions",
        "text": "If a customer decides to cancel after available options have been explained, the cancellation process should be handled according to policy."
    }
]

knowledge_base.extend(retention_records)

print("Total Knowledge Base documents:", len(knowledge_base))

Total Knowledge Base documents: 84


In [36]:
support_procedure_records = [
    {
        "category": "Support Procedures",
        "title": "Verify Customer Account",
        "text": "Support staff should verify the customer's account information before handling account-specific requests."
    },
    {
        "category": "Support Procedures",
        "title": "Understand the Issue",
        "text": "Support staff should identify the customer's problem and collect the relevant details before troubleshooting."
    },
    {
        "category": "Support Procedures",
        "title": "Check Previous Interactions",
        "text": "Previous support interactions can be reviewed to determine whether the customer has experienced the same issue before."
    },
    {
        "category": "Support Procedures",
        "title": "Provide Troubleshooting",
        "text": "Support staff should provide appropriate troubleshooting steps based on the customer's reported problem."
    },
    {
        "category": "Support Procedures",
        "title": "Confirm Resolution",
        "text": "After troubleshooting, support staff should confirm whether the customer's issue has been resolved."
    },
    {
        "category": "Support Procedures",
        "title": "Escalate Unresolved Issues",
        "text": "Issues that cannot be resolved through standard troubleshooting should be escalated to the appropriate team."
    },
    {
        "category": "Support Procedures",
        "title": "Document Support Cases",
        "text": "Important customer issues, actions taken, and outcomes should be documented in the support record."
    },
    {
        "category": "Support Procedures",
        "title": "Handle Billing Issues",
        "text": "Billing-related issues should be reviewed using the customer's account and billing information before providing a resolution."
    },
    {
        "category": "Support Procedures",
        "title": "Handle Cancellation Requests",
        "text": "Cancellation requests should be verified and processed according to the applicable cancellation policy."
    },
    {
        "category": "Support Procedures",
        "title": "Close Resolved Tickets",
        "text": "A support ticket can be closed after the issue has been resolved and the necessary information has been recorded."
    }
]

knowledge_base.extend(support_procedure_records)

print("Total Knowledge Base documents:", len(knowledge_base))

Total Knowledge Base documents: 94


In [37]:
business_rule_records = [
    {
        "category": "Business Rules",
        "title": "Account Verification",
        "text": "Account-specific information should only be discussed after the customer's account has been verified."
    },
    {
        "category": "Business Rules",
        "title": "Customer Privacy",
        "text": "Customer information should be handled carefully and only used for legitimate support and business purposes."
    },
    {
        "category": "Business Rules",
        "title": "Accurate Information",
        "text": "Support staff should provide customers with accurate information about products, services, billing, and policies."
    },
    {
        "category": "Business Rules",
        "title": "Policy Compliance",
        "text": "Customer requests should be handled according to the applicable company policies and business rules."
    },
    {
        "category": "Business Rules",
        "title": "Escalation Rule",
        "text": "Requests outside the authority of support staff should be escalated to the appropriate team."
    },
    {
        "category": "Business Rules",
        "title": "Document Important Actions",
        "text": "Important account changes, resolutions, and customer decisions should be documented in the appropriate record."
    },
    {
        "category": "Business Rules",
        "title": "Refund Approval",
        "text": "Refunds and billing adjustments should follow the required approval rules before they are processed."
    },
    {
        "category": "Business Rules",
        "title": "Cancellation Processing",
        "text": "Cancellation requests should be processed only after the required customer and account information has been verified."
    },
    {
        "category": "Business Rules",
        "title": "Consistent Support",
        "text": "Similar customer issues should be handled consistently using the applicable support procedures and policies."
    },
    {
        "category": "Business Rules",
        "title": "Record Resolution",
        "text": "Resolved customer issues should have the resolution and relevant actions recorded before the support case is closed."
    }
]

knowledge_base.extend(business_rule_records)

print("Total Knowledge Base documents:", len(knowledge_base))

Total Knowledge Base documents: 104


In [38]:
kb_df = pd.DataFrame(knowledge_base)

print("Knowledge Base shape:", kb_df.shape)
print("Documents by category:")
print(kb_df['category'].value_counts())

Knowledge Base shape: (104, 3)
Documents by category:
category
Service Policy          13
Customer Support FAQ    13
Contract Information    13
Cancellation Policy     13
Billing Policy          13
Retention Guidelines    13
Support Procedures      13
Business Rules          13
Name: count, dtype: int64


In [39]:
kb_df.to_csv("../data/knowledge_base/knowledge_base.csv",index=False)

chunking

In [40]:
def chunk_text(text, chunk_size=50):
    words = text.split()
    
    chunks = []
    
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    return chunks

In [41]:
kb_chunks = []

for _, row in kb_df.iterrows():
    chunks = chunk_text(row['text'])
    
    for chunk in chunks:
        kb_chunks.append({
            "category": row['category'],
            "title": row['title'],
            "text": chunk
        })

kb_chunks_df = pd.DataFrame(kb_chunks)

print("Original documents:", len(kb_df))
print("Total chunks:", len(kb_chunks_df))

Original documents: 104
Total chunks: 104


In [42]:
kb_chunks_df.head(10)

,category,title,text
0,Service Policy,Support Channels,Customers can contact customer support through...
1,Service Policy,Business Hours,Standard customer support requests are normall...
2,Service Policy,Urgent Issues,Urgent service interruptions may receive highe...
3,Customer Support FAQ,Required Information,"Customers should provide their customer ID, or..."
4,Customer Support FAQ,Product Not Working,"If a product is not working, customers should ..."
5,Customer Support FAQ,Software Problems,"When reporting a software problem, customers s..."
6,Contract Information,Contract Review,Customers should review their applicable contr...
7,Contract Information,Month-to-Month Contract,Month-to-month contracts generally provide gre...
8,Contract Information,Contract Changes,Contract changes must be processed according t...
9,Cancellation Policy,Cancellation Request,Customers who want to cancel a service should ...


vectorization

In [43]:
model = SentenceTransformer("../models/all-MiniLM-L6-v2")

kb_vectors = model.encode(kb_chunks_df['text'].tolist()).astype('float32')

print("Number of chunks:", len(kb_chunks_df))
print("Vector shape:", kb_vectors.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8715.57it/s]


Number of chunks: 104
Vector shape: (104, 384)


FIASS

In [44]:
dimension = kb_vectors.shape[1]

kb_index = faiss.IndexFlatL2(dimension)

kb_index.add(kb_vectors)

print("FAISS vectors:", kb_index.ntotal)
print("Vector dimension:", kb_index.d)

FAISS vectors: 104
Vector dimension: 384


In [45]:
faiss.write_index(kb_index,"../models/kb_faiss.index")

kb_chunks_df.to_pickle("../models/kb_chunks.pkl")

retrive and testing the extraction FIASS

In [46]:
kb_index = faiss.read_index("../models/kb_faiss.index")
kb_chunks_df = pd.read_pickle("../models/kb_chunks.pkl")

In [47]:
query = "How can I cancel my subscription?"

query_vector = model.encode([query]).astype('float32')

distances, indices = kb_index.search(query_vector, 5)

print("Retrieved chunks:")
for i in indices[0]:
    print("\nCategory:", kb_chunks_df.iloc[i]['category'])
    print("Title:", kb_chunks_df.iloc[i]['title'])
    print("Text:", kb_chunks_df.iloc[i]['text'])

Retrieved chunks:

Category: Cancellation Policy
Title: Account Verification
Text: The customer account should be verified before a cancellation request is processed.

Category: Cancellation Policy
Title: Verify Account
Text: The customer account should be verified before a cancellation request is processed.

Category: Business Rules
Title: Cancellation Processing
Text: Cancellation requests should be processed only after the required customer and account information has been verified.

Category: Cancellation Policy
Title: Cancellation Request
Text: Customers who want to cancel a service should contact customer support through an approved support channel.

Category: Cancellation Policy
Title: Cancellation Request
Text: Customers who want to cancel a service should contact customer support through an approved support channel.


context creation on top 

In [48]:
query = "How can I cancel my subscription?"

query_vector = model.encode([query]).astype('float32')

distances, indices = kb_index.search(query_vector, 5)

context = ""

for i in indices[0]:
    context += f"""
Category: {kb_chunks_df.iloc[i]['category']}
Title: {kb_chunks_df.iloc[i]['title']}
Information: {kb_chunks_df.iloc[i]['text']}
---
"""

print(context)


Category: Cancellation Policy
Title: Account Verification
Information: The customer account should be verified before a cancellation request is processed.
---

Category: Cancellation Policy
Title: Verify Account
Information: The customer account should be verified before a cancellation request is processed.
---

Category: Business Rules
Title: Cancellation Processing
Information: Cancellation requests should be processed only after the required customer and account information has been verified.
---

Category: Cancellation Policy
Title: Cancellation Request
Information: Customers who want to cancel a service should contact customer support through an approved support channel.
---

Category: Cancellation Policy
Title: Cancellation Request
Information: Customers who want to cancel a service should contact customer support through an approved support channel.
---



LLM

In [49]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(model_name,torch_dtype="auto")

print("Tokenizer loaded")
print("LLM loaded")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 8146.81it/s]


Tokenizer loaded
LLM loaded


In [50]:
prompt = "What is customer support?"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = llm.generate(**inputs,max_new_tokens=50)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

What is customer support? Customer support, also known as customer service or client services, refers to the actions taken by a company to assist its customers in resolving any issues they may have. This can include providing technical assistance for software or hardware products, offering advice on product usage,


In [ ]:
prompt = f"""You are a customer support assistant.

Your task is to answer the user's question using ONLY the information in the context.

Context:
{context}

User Question:
{query}

Answer:"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = llm.generate(**inputs,max_new_tokens=100)

answer = tokenizer.decode(outputs[0],skip_special_tokens=True)

print(answer)


You are a customer support assistant.

Your task is to answer the user's question using ONLY the information in the context.

Context:

Category: Cancellation Policy
Title: Account Verification
Information: The customer account should be verified before a cancellation request is processed.
---

Category: Cancellation Policy
Title: Verify Account
Information: The customer account should be verified before a cancellation request is processed.
---

Category: Business Rules
Title: Cancellation Processing
Information: Cancellation requests should be processed only after the required customer and account information has been verified.
---

Category: Cancellation Policy
Title: Cancellation Request
Information: Customers who want to cancel a service should contact customer support through an approved support channel.
---

Category: Cancellation Policy
Title: Cancellation Request
Information: Customers who want to cancel a service should contact customer support through an approved support cha

prompt engineering

with context

In [ ]:
prompt = f"""Answer the user's question using the provided context.

Context:
{context}

User Question:
{query}

Answer:"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = llm.generate(**inputs,max_new_tokens=100)

answer = tokenizer.decode(outputs[0],skip_special_tokens=True)

print(answer)


Answer the user's question using the provided context.

Context:

Category: Cancellation Policy
Title: Account Verification
Information: The customer account should be verified before a cancellation request is processed.
---

Category: Cancellation Policy
Title: Verify Account
Information: The customer account should be verified before a cancellation request is processed.
---

Category: Business Rules
Title: Cancellation Processing
Information: Cancellation requests should be processed only after the required customer and account information has been verified.
---

Category: Cancellation Policy
Title: Cancellation Request
Information: Customers who want to cancel a service should contact customer support through an approved support channel.
---

Category: Cancellation Policy
Title: Cancellation Request
Information: Customers who want to cancel a service should contact customer support through an approved support channel.
---


User Question:
How can I cancel my subscription?

Answer:


with chrun tenure and contract type

In [ ]:
prompt = f"""You are a customer intelligence assistant.Use only the information provided.

Customer:
Tenure: 5 months
Contract: Month-to-month
Monthly Charges: 95.20

Churn:
Probability: 0.82

SHAP:
Contract type, Tenure, Monthly charges

Knowledge:
{context}

Question:
{query}

Answer:"""

In [55]:
inputs = tokenizer(prompt, return_tensors="pt")

outputs = llm.generate(**inputs,max_new_tokens=100)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)


You are a customer intelligence assistant.
Use only the information provided.

Customer:
Tenure: 5 months
Contract: Month-to-month
Monthly Charges: 95.20

Churn:
Probability: 0.82

SHAP:
Contract type, Tenure, Monthly charges

Knowledge:

Category: Cancellation Policy
Title: Account Verification
Information: The customer account should be verified before a cancellation request is processed.
---

Category: Cancellation Policy
Title: Verify Account
Information: The customer account should be verified before a cancellation request is processed.
---

Category: Business Rules
Title: Cancellation Processing
Information: Cancellation requests should be processed only after the required customer and account information has been verified.
---

Category: Cancellation Policy
Title: Cancellation Request
Information: Customers who want to cancel a service should contact customer support through an approved support channel.
---

Category: Cancellation Policy
Title: Cancellation Request
Information:

strict prompt

In [ ]:
prompt = f"""Answer the question using ONLY the provided information.Do not add or assume anything.
Return only the answer.

Customer:
Tenure: 5 months
Contract: Month-to-month
Monthly Charges: 95.20

Churn probability: 0.82

SHAP:
Contract type, Tenure, Monthly charges

Knowledge:
{context}

Question:
{query}

Answer:"""

In [59]:
inputs = tokenizer(prompt, return_tensors="pt")

outputs = llm.generate(**inputs,max_new_tokens=100)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)


Answer the question using ONLY the provided information.Do not add or assume anything.
Return only the answer.

Customer:
Tenure: 5 months
Contract: Month-to-month
Monthly Charges: 95.20

Churn probability: 0.82

SHAP:
Contract type, Tenure, Monthly charges

Knowledge:

Category: Cancellation Policy
Title: Account Verification
Information: The customer account should be verified before a cancellation request is processed.
---

Category: Cancellation Policy
Title: Verify Account
Information: The customer account should be verified before a cancellation request is processed.
---

Category: Business Rules
Title: Cancellation Processing
Information: Cancellation requests should be processed only after the required customer and account information has been verified.
---

Category: Cancellation Policy
Title: Cancellation Request
Information: Customers who want to cancel a service should contact customer support through an approved support channel.
---

Category: Cancellation Policy
Title: C